# Netflix'te Ne İzlesem? Basit Bir Öneri Sistemi

[![Colab'da Aç](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haticeozbolat01/pop-kultur-kodluyorum/blob/main/07-ne-izlesem-oneri-sistemi.ipynb)

Bu not defteri, haticeozbolat.com'daki **Netflix'te Ne İzlesem? Basit Bir Öneri Sistemi Yazmak** yazısının kodlarını içeriyor. Yazı: [Yazıyı oku]([YAZI LİNKİ])

**Veri hakkında:** Katalogdaki dizi adları gerçek, ama etiketleri ben verdim; öznel ve eksik olabilirler. Netflix'in verisi, görselleri ya da logosu kullanılmıyor. Netflix'in kendi öneri sisteminin nasıl çalıştığını taklit etmiyor; öneri sistemlerinin temel fikrini gösteriyor.

Hücreleri sırayla **Shift + Enter** ile çalıştır.

## 1. Küçük bir katalog: her dizi bir etiket kümesi

In [ ]:
KATALOG = {
    "Stranger Things": {"bilim kurgu", "korku", "gençlik", "80'ler", "gizem"},
    "Dark": {"bilim kurgu", "gizem", "zaman yolculuğu", "aile", "Almanya"},
    "Black Mirror": {"bilim kurgu", "teknoloji", "antoloji", "distopya"},
    "3 Body Problem": {"bilim kurgu", "gizem", "uzay", "bilim"},
    "Love, Death & Robots": {"bilim kurgu", "animasyon", "antoloji", "teknoloji"},
    "Squid Game": {"gerilim", "hayatta kalma", "Kore", "distopya"},
    "Alice in Borderland": {"gerilim", "hayatta kalma", "Japonya", "gizem"},
    "La Casa de Papel": {"soygun", "gerilim", "İspanya", "suç"},
    "Lupin": {"soygun", "gizem", "Fransa", "suç"},
    "Narcos": {"suç", "biyografi", "drama", "gerçek hikâye"},
    "Mindhunter": {"suç", "psikolojik", "gerilim", "gerçek hikâye"},
    "Ozark": {"suç", "gerilim", "drama", "aile"},
    "The Crown": {"drama", "tarih", "biyografi", "gerçek hikâye"},
    "The Queen's Gambit": {"drama", "dönem", "satranç", "gençlik"},
    "Bridgerton": {"romantik", "dönem", "drama"},
    "Wednesday": {"gizem", "komedi", "gençlik", "fantastik"},
    "The Witcher": {"fantastik", "aksiyon", "macera"},
    "Arcane": {"animasyon", "fantastik", "aksiyon", "aile"},
    "Brooklyn Nine-Nine": {"komedi", "polisiye", "iş yeri"},
    "Crash Landing on You": {"romantik", "komedi", "Kore", "drama"},
}

print(len(KATALOG), "dizi")

## 2. İki dizi ne kadar benziyor? Jaccard benzerliği

In [ ]:
def jaccard(a, b):
    """İki kümenin ortak eleman sayısının, toplam farklı eleman sayısına oranı."""
    return len(a & b) / len(a | b)

dark = KATALOG["Dark"]
st = KATALOG["Stranger Things"]
print("Ortak:", dark & st)
print("Toplam:", len(dark | st))
print("Benzerlik:", round(jaccard(dark, st), 2))

## 3. "Bunu sevdiysen" önerileri

In [ ]:
def benzerleri(dizi, n=5):
    hedef = KATALOG[dizi]
    puanlar = {d: jaccard(hedef, etiket) for d, etiket in KATALOG.items() if d != dizi}
    return sorted(puanlar.items(), key=lambda x: -x[1])[:n]

for dizi in ["Dark", "Squid Game", "The Crown"]:
    print(f"\n{dizi} sevdiysen:")
    for ad, puan in benzerleri(dizi):
        print(f"  {ad:<22} {puan:.2f}")

## 4. Grafik

In [ ]:
import matplotlib.pyplot as plt

sonuc = benzerleri("Dark")
adlar = [a for a, _ in sonuc][::-1]
puanlar = [p for _, p in sonuc][::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(adlar, puanlar, color="#7A4290", height=0.6)
for y, p in enumerate(puanlar):
    ax.text(p + 0.005, y, f"{p:.2f}", va="center", color="#333333")
ax.set_xlabel("Jaccard benzerliği")
ax.set_xlim(0, max(puanlar) * 1.2)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Dark'ı sevenlere ilk 5 öneri", loc="left")
plt.tight_layout()
plt.show()

## 5. Birden fazla diziden zevk profili çıkarmak

In [ ]:
from collections import Counter

def profil_oner(begenilenler, n=5):
    profil = Counter()
    for dizi in begenilenler:
        profil.update(KATALOG[dizi])
    puanlar = {
        d: sum(profil[e] for e in etiket) / len(etiket)
        for d, etiket in KATALOG.items() if d not in begenilenler
    }
    return profil, sorted(puanlar.items(), key=lambda x: -x[1])[:n]

profil, oneriler = profil_oner(["Dark", "Black Mirror", "Mindhunter"])
print("Zevk profili:", profil.most_common(5))
for ad, puan in oneriler:
    print(f"  {ad:<22} {puan:.2f}")

## 6. Kendi listenle dene

Aşağıdaki listeye katalogdan sevdiğin dizileri yaz.

In [ ]:
begendiklerim = ["Wednesday", "The Queen's Gambit"]

_, oneriler = profil_oner(begendiklerim)
for ad, puan in oneriler:
    print(f"  {ad:<22} {puan:.2f}")

## Kendin dene

1. Kataloğa sevdiğin üç diziyi kendi etiketleriyle ekle.
2. The Crown için Narcos neden en benzer çıkıyor? Etiketleri değiştirerek bunu düzelt.
3. `profil_oner` içinde etiket sayısına bölmeyi kaldır. Sonuçlar nasıl değişiyor, neden?
4. Jaccard yerine yalnızca ortak etiket sayısını (`len(a & b)`) kullan ve sonuçları karşılaştır.